In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys
import os
import subprocess
import argparse

def select_tsv_file():
    """
    Searches for TSV files in the 'data' subdirectory of the current directory
    and allows the user to select one interactively.
    """
    search_dir = os.path.join(os.getcwd(), "data")
    
    if not os.path.isdir(search_dir):
        sys.exit(f"Error: Directory not found: {search_dir}")
        
    try:
        all_files = os.listdir(search_dir)
        tsv_files = [os.path.join(search_dir, f) for f in all_files if f.lower().endswith('.tsv') and os.path.isfile(os.path.join(search_dir, f))]
    except OSError as e:
        sys.exit(f"Error: Cannot read directory {search_dir}: {e}")

    if not tsv_files:
        sys.exit("Error: No TSV files found in data subdirectory")

    print("Available TSV files:")
    for i, tsv_file in enumerate(tsv_files):
        print(f"{i + 1:2d}: {tsv_file}")

    while True:
        try:
            choice_str = input(f"\nEnter the number of the TSV file to process (1-{len(tsv_files)}): ")
            choice = int(choice_str)
            if 1 <= choice <= len(tsv_files):
                return tsv_files[choice - 1]
            else:
                print(f"Error: Please enter a number between 1 and {len(tsv_files)}.")
        except ValueError:
            print("Error: Invalid input. Please enter a number.")
        except (EOFError, KeyboardInterrupt):
            sys.exit("\nOperation cancelled by user.")


def main():
    """
    Main function to orchestrate the pipeline execution.
    """
    parser = argparse.ArgumentParser(
        description="Runs the microbiome analysis pipeline.",
        epilog="If a TSV file is not provided as an argument, the script will enter interactive mode to select a file."
    )
    parser.add_argument("tsv_file", nargs="?", default=None, help="Path to the target TSV file for processing.")
    args, unknown = parser.parse_known_args()

    # --- Determine the input file ---
    # Use the argument only if it was provided and is not a Jupyter kernel file.
    tsv_file = ""
    is_kernel_file = args.tsv_file and 'kernel-' in args.tsv_file and args.tsv_file.endswith('.json')
    
    if args.tsv_file and not is_kernel_file:
        # A legitimate file path was passed as an argument.
        tsv_file = args.tsv_file
    else:
        # If no argument was provided, or if it was a Jupyter kernel file, enter interactive mode.
        if is_kernel_file:
            print("! Detected Jupyter kernel argument. Ignoring it and switching to interactive mode.")
        tsv_file = select_tsv_file()

    if not os.path.isfile(tsv_file):
        sys.exit(f"Error: Cannot find tsv file at {tsv_file}")

    sample_name = os.path.splitext(os.path.basename(tsv_file))[0]
    
    try:
        # Get the directory where this script is located.
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        # Fallback for interactive environments (e.g., Jupyter) where __file__ is not defined.
        base_dir = os.getcwd()
    
    print(f"Debug: base_dir = {base_dir}")
    print(f"Debug: sample_name = {sample_name}")

    output_dir = os.path.join(base_dir, sample_name)
    print(f"Debug: output_dir = {output_dir}")
    os.makedirs(output_dir, exist_ok=True)

    done_dir = os.path.join(output_dir, "done")
    os.makedirs(done_dir, exist_ok=True)

    # === Step Definition ===
    # Define the pipeline steps. The 'script' path can point to a .py or .pl file.
    # This list can be mixed (e.g., some Python, some Perl) to allow for incremental migration.
    steps = [
        {'name': 'assembly',     'script': 'scripts/microbiome_assembly.py',      'depends_on': []},
        {'name': 'get_datasets', 'script': 'scripts/microbiome_get_datasets.py',  'depends_on': ['assembly']},
        {'name': 'extract',      'script': 'scripts/microbiome_extract.py',       'depends_on': ['get_datasets']},
    ]

    # === Execution Loop ===
    done = {}
    for step in steps:
        step_name = step['name']
        flag = os.path.join(done_dir, f"{step_name}.done")

        if os.path.exists(flag):
            print(f"? Skip {step_name} (done)")
            done[step_name] = True
            continue

        # Check for dependencies.
        for dep in step['depends_on']:
            if not done.get(dep):
                sys.exit(f"? Cannot run {step_name} before {dep} is completed")

        print(f"?? Running {step_name} for sample: {sample_name}")

        script_path = os.path.join(base_dir, step['script'])
        cmd = []

        # --- Dynamic Command Generation ---
        # Determine the correct interpreter (python3 or perl) based on the script's file extension.
        interpreter = ''
        if script_path.endswith('.py'):
            interpreter = 'python3'  # This can be changed to 'python' depending on the environment.
        elif script_path.endswith('.pl'):
            interpreter = 'perl'
        else:
            sys.exit(f"Error: Unknown script type for {script_path}")

        # Build the command list with arguments specific to each step.
        if step_name in ['get_datasets', 'extract']:
            # These steps take the output directory as an argument.
            cmd = [interpreter, script_path, output_dir]
        else: # 'assembly' step
            # This step takes the input TSV and output directory as arguments.
            cmd = [interpreter, script_path, tsv_file, output_dir]
        
        print(f"  Command: {' '.join(cmd)}")

        try:
            # To display stdout in real-time, more advanced techniques like subprocess.Popen
            # would be needed. This simpler approach captures output and prints it upon completion.
            result = subprocess.run(cmd, check=True, text=True, capture_output=True)
            if result.stdout:
                print(result.stdout) # Print the successful output of the subscript.
        except FileNotFoundError:
             sys.exit(f"? Command not found. Is '{interpreter}' in your PATH? Failed step: {step_name}")
        except subprocess.CalledProcessError as e:
            print(f"\n--- Error Output from {step_name} ---", file=sys.stderr)
            print(e.stderr, file=sys.stderr)
            print("------------------------------------", file=sys.stderr)
            sys.exit(f"? Step {step_name} failed (exit: {e.returncode})")

        # Create a flag file to mark the step as completed.
        try:
            with open(flag, 'w') as f:
                f.write("done\n")
        except IOError as e:
            sys.exit(f"? Cannot write {flag}: {e}")
        
        done[step_name] = True
        print(f"? Completed {step_name}")

    print(f"\n?? All steps completed successfully for {sample_name}")

if __name__ == "__main__":
    main()

! Detected Jupyter kernel argument. Ignoring it and switching to interactive mode.
Available TSV files:
 1: /home/3a18004/BLING/data/level-7.tsv
 2: /home/3a18004/BLING/data/pain_sensitivity.tsv
 3: /home/3a18004/BLING/data/silva_pain_sensitivity.tsv
 4: /home/3a18004/BLING/data/depression.tsv

Enter the number of the TSV file to process (1-4): 1
Debug: base_dir = /home/3a18004/BLING
Debug: sample_name = level-7
Debug: output_dir = /home/3a18004/BLING/level-7
?? Running assembly for sample: level-7
  Command: python3 /home/3a18004/BLING/scripts/microbiome_assembly.py /home/3a18004/BLING/data/level-7.tsv /home/3a18004/BLING/level-7
? Output Directory: /home/3a18004/BLING/level-7
--- Executing generated commands ---
Do: datasets summary genome taxon 447020 --as-json-lines --reference | dataformat tsv genome --fields accession,organism-name,assminfo-level,organism-tax-id > Genus_Adlercreutzia/Genus_Adlercreutzia.tsv
Do: datasets summary genome taxon 171549 --as-json-lines --reference | da

? Genome data will be stored in shared directory: /home/3a18004/BLING/data/rawgbff
Processing accession file: /home/3a18004/BLING/level-7/Genus_Adlercreutzia/Genus_Adlercreutzia.tsv
Processing accession file: /home/3a18004/BLING/level-7/Order_Bacteroidales/Order_Bacteroidales.tsv
Processing accession file: /home/3a18004/BLING/level-7/Genus_Bacteroides/Genus_Bacteroides.tsv
Processing accession file: /home/3a18004/BLING/level-7/Species_acidifaciens/Species_acidifaciens.tsv
Processing accession file: /home/3a18004/BLING/level-7/Genus_Parabacteroides/Genus_Parabacteroides.tsv
Processing accession file: /home/3a18004/BLING/level-7/Species_distasonis/Species_distasonis.tsv
Processing accession file: /home/3a18004/BLING/level-7/Genus_Prevotella/Genus_Prevotella.tsv
Processing accession file: /home/3a18004/BLING/level-7/Family_Rikenellaceae/Family_Rikenellaceae.tsv
Processing accession file: /home/3a18004/BLING/level-7/Family_Muribaculaceae/Family_Muribaculaceae.tsv
Processing accession file:

In [1]:
#!/usr/bin/env python3
import sys
import os
import glob
import argparse
import subprocess
from pathlib import Path

def select_target_directory():
    """
    Interactively select a target directory if one is not provided via arguments.
    It searches for subdirectories in the current location that contain 'commands.txt'.
    """
    print("Searching for target directories containing 'commands.txt'...")
    
    potential_dirs = [d.name for d in Path.cwd().iterdir() if d.is_dir() and (d / 'commands.txt').is_file()]

    if not potential_dirs:
        print("Error: No target directories containing 'commands.txt' were found.", file=sys.stderr)
        sys.exit(1)

    print("\nPlease choose the directory to analyze:")
    for i, directory in enumerate(potential_dirs):
        print(f" {i + 1:2d}: {directory}")
    
    try:
        choice_str = input(f"\nEnter number (1-{len(potential_dirs)}): ")
        choice = int(choice_str.strip())
        if 1 <= choice <= len(potential_dirs):
            chosen_dir = potential_dirs[choice - 1]
            print(f"-> Target directory set to '{chosen_dir}'.")
            return chosen_dir
        else:
            raise ValueError("Choice out of range.")
    except (ValueError, EOFError, KeyboardInterrupt):
        print("\nError: Invalid selection or operation cancelled.", file=sys.stderr)
        sys.exit(1)

def main():
    """
    Main function to run the post-extraction pipeline steps.
    """
    # === Argument Parsing ===
    parser = argparse.ArgumentParser(
        description="Run post-extraction analysis steps (InterProScan, etc.).",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )
    parser.add_argument('--base-dir', type=str,
                        help="The base directory for a specific sample, containing previous step results.")
    parser.add_argument('--cpu', type=int, default=8,
                        help="Number of CPU cores to use for applicable steps.")
    parser.add_argument('--appl', type=str, default='TIGRFAM,SUPERFAMILY,PANTHER,Gene3D,Hamap,SMART,CDD,PRINTS,Pfam,ProSitePatterns,ProSiteProfiles',
                        help="Comma-separated list of applications for InterProScan.")
    
    args, unknown = parser.parse_known_args()
    
    # If --base-dir is not provided, enter interactive selection mode.
    if not args.base_dir:
        args.base_dir = select_target_directory()
        
    # === Path Configuration ===
    # Project root is the directory from where the script is launched.
    project_root_dir = Path.cwd()
    # The sample directory path should be absolute.
    sample_base_dir = Path(args.base_dir).resolve()
    
    done_dir = sample_base_dir / 'done'
    done_dir.mkdir(exist_ok=True)
    
    # === Step Definition ===
    steps = [
        {
            'name': 'interpro', 'script': 'scripts/microbiome_run_interpro.py',
            'depends_on': ['extract'],
            'pass_opts': {'base_dir': True, 'cpu': True, 'appl': True}
        },
        {
            'name': 'interval', 'script': 'scripts/microbiome_interval.py',
            'depends_on': ['interpro'],
            'pass_opts': {'base_dir': True}
        },
        {
            'name': 'merge_GO', 'script': 'scripts/microbiome_merge_GO.py',
            'depends_on': ['interval'],
            'pass_opts': {'base_dir': True}
        },
    ]
    
    # === Execution Management ===
    completed_steps = {}

    # Pre-check for the completion of the 'extract' step from the previous pipeline.
    if (done_dir / "extract.done").is_file():
        completed_steps['extract'] = True

    for step in steps:
        step_name = step['name']
        flag_file = done_dir / f"{step_name}.done"

        if flag_file.exists():
            print(f"✅ Skip {step_name} (done)")
            completed_steps[step_name] = True
            continue

        for dep in step['depends_on']:
            if not completed_steps.get(dep):
                print(f" Cannot run {step_name} before {dep} is completed", file=sys.stderr)
                sys.exit(1)
                
        print(f"\n === STEP: {step_name} ===")

        # Dynamically build the command to be executed.
        script_path = project_root_dir / step['script']
        interpreter = 'python3' if str(script_path).endswith('.py') else 'perl'
        cmd = [interpreter, str(script_path)]

        # Add options to the command based on the 'pass_opts' definition.
        pass_opts = step.get('pass_opts', {})
        if pass_opts.get('base_dir'):
            cmd.extend(['--base-dir', str(sample_base_dir)])
        if pass_opts.get('cpu'):
            cmd.extend(['--cpu', str(args.cpu)])
        if pass_opts.get('appl'):
            cmd.extend(['--appl', args.appl])
            
        print(f"   Running: {' '.join(cmd)}")
        try:
            # Execute the command. check=True will raise an exception on non-zero exit codes.
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f" Step {step_name} failed (exit: {e.returncode})", file=sys.stderr)
            sys.exit(1)
        except FileNotFoundError:
            print(f" Command not found. Is '{interpreter}' in your PATH? Failed step: {step_name}", file=sys.stderr)
            sys.exit(1)


        # Create a flag file to mark the step as completed, and write "done" into it.
        try:
            flag_file.write_text("done\n")
        except IOError as e:
            print(f" Cannot write {flag_file}: {e}", file=sys.stderr)
            sys.exit(1)

        
        completed_steps[step_name] = True
        print(f" Completed {step_name}")

    print("\n All steps completed successfully.")

if __name__ == "__main__":
    main()

Searching for target directories containing 'commands.txt'...

Please choose the directory to analyze:
  1: microbiome
  2: pain_sensitivity
  3: silva_pain_sensitivity
  4: depression
  5: level-7

Enter number (1-5): 5
-> Target directory set to 'level-7'.

 === STEP: interpro ===
   Running: python3 /home/3a18004/BLING/scripts/microbiome_run_interpro.py --base-dir /home/3a18004/BLING/level-7 --cpu 8 --appl TIGRFAM,SUPERFAMILY,PANTHER,Gene3D,Hamap,SMART,CDD,PRINTS,Pfam,ProSitePatterns,ProSiteProfiles
Using InterProScan at: /home/3a18004/BLING/tools/interproscan-5.75-106.0/interproscan.sh
[?] Found existing InterProScan output for GCF_004340205.1 -> skipping.
[?] Found existing InterProScan output for GCF_012317185.1 -> skipping.
[?] Found existing InterProScan output for GCF_000621585.1 -> skipping.
[?] Found existing InterProScan output for GCF_001559865.1 -> skipping.
[?] Found existing InterProScan output for GCF_900103835.1 -> skipping.
[?] Found existing InterProScan output for 

[?] Found existing InterProScan output for GCF_030586845.1 -> skipping.
[?] Found existing InterProScan output for GCF_030587225.1 -> skipping.
[?] Found existing InterProScan output for GCF_030586865.1 -> skipping.
[?] Found existing InterProScan output for GCF_030586805.1 -> skipping.
[?] Found existing InterProScan output for GCF_038024965.1 -> skipping.
[?] Found existing InterProScan output for GCF_030586975.1 -> skipping.
[?] Found existing InterProScan output for GCF_018024475.1 -> skipping.
[?] Found existing InterProScan output for GCF_000026205.1 -> skipping.
[?] Found existing InterProScan output for GCF_032884545.1 -> skipping.
[?] Found existing InterProScan output for GCF_016876135.2 -> skipping.
[?] Found existing InterProScan output for GCF_003263355.1 -> skipping.
[?] Found existing InterProScan output for GCF_046896035.1 -> skipping.
[?] Found existing InterProScan output for GCF_001189415.1 -> skipping.
[?] Found existing InterProScan output for GCF_007821455.1 -> sk

[?] Found existing InterProScan output for GCF_013315815.1 -> skipping.
[?] Found existing InterProScan output for GCF_014384785.1 -> skipping.
[?] Found existing InterProScan output for GCF_018919205.1 -> skipping.
[?] Found existing InterProScan output for GCF_014297285.1 -> skipping.
[?] Found existing InterProScan output for GCF_036321785.1 -> skipping.
[?] Found existing InterProScan output for GCF_000178115.2 -> skipping.
[?] Found existing InterProScan output for GCF_020687345.1 -> skipping.
[?] Found existing InterProScan output for GCF_010509575.1 -> skipping.
[?] Found existing InterProScan output for GCF_040096475.1 -> skipping.
[?] Found existing InterProScan output for GCF_022180365.1 -> skipping.
[?] Found existing InterProScan output for GCF_003287455.1 -> skipping.
[?] Found existing InterProScan output for GCF_040096575.1 -> skipping.
[?] Found existing InterProScan output for GCF_002549775.1 -> skipping.
[?] Found existing InterProScan output for GCF_040094125.1 -> sk

KeyboardInterrupt: 